# 欢迎来到非常忙碌的第 8 周文件夹

## 本周有很多事要做！

我们会比平时更快推进，尤其是你已经逐渐成为熟练的 LLM 工程师。

# The Price is Right

## 第 8 周日程安排

第 1 天：Modal.com 与 SpecialistAgent  
第 2 天：RAG、FrontierAgent、Ensemble Agent  
第 3 天：ScannerAgent、MessengerAgent  
第 4 天：AutonomousPlannerAgent 与 DealAgentFramework  
第 5 天：The Price Is Right 终章




<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#b22;">本周特别重要：请拉取最新代码</h2>
            <span style="color:#b22;">我在持续改进这些实验，补充更多示例和练习。
            每周开始时，都值得确认你已拿到最新代码。<br/>
            如果不确定如何执行 <code>git pull</code>，请查看 Guides 文件夹中的 Guide 3。
            </span>
        </td>
    </tr>
</table>

In [ ]:
# 导入：Modal（云端 GPU 部署）、预处理 Agent、环境变量
# 本周目标：把微调模型部署到 Modal，并封装成 Specialist Agent

import os
import locale
import modal
from agents.preprocessor import Preprocessor
from dotenv import load_dotenv
load_dotenv(override=True)

In [ ]:
# 检查你的电脑能否输出特殊字符，确认这里输出的是 UTF-8

print(locale.getpreferredencoding())  # Should print 'UTF-8'

In [ ]:
# 强制 Python 用 UTF-8，避免终端/日志里中文或特殊字符乱码

os.environ["PYTHONIOENCODING"] = "utf-8"

# 设置 Modal token

## 重要——请仔细阅读并按说明操作！

首先请访问：https://modal.com

注册一个账号。然后点击右上角的头像菜单，选择「Settings」

接着在左侧边栏点击「API Tokens」，再点击「New Token」。

你会得到类似这样的命令来运行：

`modal token set --token-id ak-somethinghere --token-secret as-somethinghere`

但因为我们使用 uv，真正要运行的是：

`uv run modal token set --token-id ak-somethinghere --token-secret as-somethinghere`

### 故障排除

如果遇到问题，可以尝试这 3 点：

1. 在运行 `uv run modal token set..` 之前，先试 `uv run modal token new`  

2. 来自同学 David S. 在 Windows 上的建议：

> 如果其他 Windows 用户也遇到这个问题：除了需要在命令提示符中运行 `modal token new`，你还得移动生成的 token 文件。它会把 token 文件（.modal.toml）部署到你的 Windows 用户配置文件夹。虚拟环境看不到那个位置（奇怪的是，即使我设置了环境变量并重启后也看不到）。我把该 token 文件移到实验所用的工作目录后，认证错误就消失了。

3. 用手动方式：

也可以直接把这两个密钥加到你的 .env 文件中，通常完全没问题：

```
MODAL_TOKEN_ID=ak-...
MODAL_TOKEN_SECRET=as-...
```

然后重新运行 `load_dotenv(override=True)` 以加载这些环境变量。

In [ ]:
# hello.py 里定义了最小的 Modal App，用来验证本地/远程调用

from hello import app, hello, hello_europe

In [ ]:
# .local()：在本地进程跑 Modal 函数（不占云端）

with app.run():
    reply=hello.local()
reply

In [ ]:
# .remote()：把函数发到 Modal 云端执行（第一次可能较慢，要拉镜像）

with app.run():
    reply=hello.remote()
reply

## 感谢同学 Tue H. 补充

如果你查看 hello.py，我加了一个简单的函数 hello_europe

它使用装饰器：  
`@app.function(image=image, region="eu")`

看下面的结果！更多区域相关设置见 [这里](https://modal.com/docs/guide/region-selection)

注意：指定区域会略微多消耗一些 credits。

In [ ]:
# hello_europe：演示把函数调度到欧洲区域的 Modal worker

with app.run():
    reply=hello_europe.remote()
reply

# 在继续之前——

## 我们需要把你的 HuggingFace Token 设置为 Modal 中的 secret

## 超级重要——请阅读——很多人在这里搞混！

Modal 中的 Secret 会有一个描述该 secret 的 **名称**。  
然后 secret 本身有一个 KEY 和一个 VALUE。  
我们要设置的 secret 为：  

名称：huggingface-secret  
Key：HF_TOKEN  
Value：hf_...  

## 稳妥做法：

1. 前往 modal.com，登录并进入你的 dashboard  
2. 在导航栏点击 Secrets  
3. 创建新 secret，点击 Hugging Face；这个新 secret 需要命名为 **huggingface-secret**，因为代码里就是这样引用的  
4. 把 key 填为 HF_TOKEN，value 填为你的实际 token hf_...  
5. 点击完成

### 现在回到正题：开始使用 Llama

In [ ]:
# 该 import 可能会给出关于将本地 Python 模块添加到 Image 的弃用警告
# 该警告可以安全忽略。你在其他地方也可能看到同样的警告..

from llama import app, generate

In [ ]:
# 在 Modal 上跑 Llama 续写：enable_output 可看到云端日志

with modal.enable_output():
    with app.run():
        result=generate.remote("Never gonna give you up, never gonna")
result

# 或者如果你不想被 rickroll，可以试试这个："Hey Jude, don't make it"

In [ ]:
# 临时（ephemeral）定价服务：随 notebook 启动，用完即释放

from pricer_ephemeral import app, price

In [ ]:
# 把商品描述发给云端微调模型，返回价格估算

with modal.enable_output():
    with app.run():
        result=price.remote("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")
result

In [ ]:
# Preprocessor：用 LLM 把杂乱描述整理成更干净的商品文本

preprocessor = Preprocessor()
text = preprocessor.preprocess("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")
print(text)

In [ ]:
# 也可指定其他模型（如 Groq 上的 gpt-oss）做预处理

preprocessor = Preprocessor(model_name="groq/openai/gpt-oss-20b")
text = preprocessor.preprocess("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")
print(text)

### 若希望 Preprocessor 默认使用不同模型，请把它加到你的 .env：

`PRICER_PREPROCESSOR_MODEL=groq/openai/gpt-oss-20b`

In [ ]:
# 用整理后的 text 再问一次定价模型，对比预处理是否有帮助

with modal.enable_output():
    with app.run():
        result = price.remote(text)
print(result)

In [ ]:
# 打印当前预处理后的文本，确认内容

print(text)

## 从临时（Ephemeral）App 过渡到已部署（Deployed）App

在命令行中，`uv run modal deploy xxx` 会把你的代码部署为 Deployed App

这就是你把 AI 服务封装成 API、用于生产系统的方式。

你也可以轻松构建 REST 端点；不过我们不会讲那部分，因为我们会直接从 Python 调用。

## 关于 secrets 的重要说明

在 `pricer_service.py` 和 `pricer_service2.py` 这两个文件靠近顶部的位置，你会看到类似这样的代码：  
`secrets = [modal.Secret.from_name("hf-secret")]`  
你可能需要把 `hf-secret` 改成 `huggingface-secret`，取决于 secret 在 Modal 中的配置名称。  
要核对，请访问此页面并查看第一列：  
https://modal.com/secrets/

## 给 Windows 用户的重要说明：

下一行我会在 Jupyter lab 里调用 `uv run modal deploy`；有人反馈在某些 Windows 版本上会因 Modal 向输出打印 emoji、而无法显示，从而出现奇怪的 unicode 错误。如果你遇到这种情况，请打开 Terminal 运行 `uv run modal deploy..`

In [ ]:
# 你也可以在 Terminal 中运行 "uv run modal deploy -m pricer_service"

!uv run modal deploy -m pricer_service

In [ ]:
# 连接已 deploy 的持久化服务（按名称查找远程函数）

pricer = modal.Function.from_name("pricer-service", "price")

观看它的运行过程：

https://modal.com

In [ ]:
# 这可能需要一段时间！我们很快会用更快的方法

pricer.remote(text)

In [ ]:
# 你也可以在已激活的环境中于命令行运行 "modal deploy -m pricer_service2"

!modal deploy -m pricer_service2

In [ ]:
# pricer_service2：用 Modal Cls 把模型常驻内存，避免每次冷启动加载权重
# 第二次调用会明显更快

Pricer = modal.Cls.from_name("pricer-service", "Pricer")
pricer = Pricer()
reply = pricer.price.remote(text)
print(reply)

In [ ]:
# 再调一次：模型已在容器里，响应应更快

reply = pricer.price.remote(text)
print(reply)

# 可选：保持 Modal 预热

## 一种提升 Modal pricer 服务速度的方法

第一次运行这个 Modal class 时，构建可能最多要花 10 分钟。  
之后应该会快很多……如果需要唤醒大约 30 秒，否则约 2 秒。  
如果你希望始终是 2 秒，可以通过编辑 pricer_service2.py 中的这个常量，阻止容器进入休眠：

`MIN_CONTAINERS = 0`



把它设为 1 即可保持一个容器存活。  
但请注意：这会消耗 credits！只有在你愿意让进程持续运行时才这样做。

或者，你可以运行下面的代码，它会保持预热 20 分钟，而不是 2 分钟。

### 保持预热 20 分钟后再冷却的代码：

```python
import modal
Pricer = modal.Cls.from_name("pricer-service", "Pricer")
pricer = Pricer()
pricer.update_autoscaler(scaledown_window=1200)
```

### 恢复为仅预热 2 分钟的代码：

```python
import modal
Pricer = modal.Cls.from_name("pricer-service", "Pricer")
pricer = Pricer()
pricer.update_autoscaler(scaledown_window=120)
```

## 接下来介绍我们的 Agent 类

默认会使用 Llama3.2 做预处理

如果你更想用 Groq，请像这样添加环境变量：

```
PRICER_PREPROCESSOR_MODEL=groq/openai/gpt-oss-20b
```

In [ ]:
# 打开 INFO 日志，方便观察 Agent 内部步骤

import logging
root = logging.getLogger()
root.setLevel(logging.INFO)

In [ ]:
# SpecialistAgent：封装「预处理 + Modal 微调模型定价」的完整链路

from agents.specialist_agent import SpecialistAgent

In [ ]:
# 实例化专家 Agent（内部会连上 Modal 上的 Pricer）

agent = SpecialistAgent()


In [ ]:
# 试一试：用自然语言商品描述询价

agent.price("iPhone 10")